In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import numpy as np
import csdl_alpha as csdl
from utils import *

recorder = csdl.Recorder(inline=True)
recorder.start()
angle = csdl.Variable(value=np.array([0, 0, 0]))

x = csdl.Variable(value=np.array(((0, 0, 1))).T)
mat = euler_angle_to_rotation_matrix(angle)
vec = csdl.matvec(mat, x)
dv = csdl.derivative(vec, angle)

recorder.stop()
print(x.value)
print(np.round(mat.value, 3))
print('derivative')
# print(np.round(dv_roll.value))
# print(np.round(dv_pitch.value))
# print(np.round(dv_yaw.value))
print(np.round(dv.value, 3))

[0. 0. 1.]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
derivative
[[ 0.  1.  0.]
 [-1.  0.  0.]
 [ 0.  0.  0.]]


1. Import zero-position joint transforms. 
2. Each time a joint rotates, pre-multiply a transform by rotz corresponding to its upstream actuator's rotation angle 
3. Forward Dynamics:
   1. input set of angles
   2. Create holder for cumulative transform = transform_c
   3. For each angle_i in angle:
      1. transform_c = transform_c * rot_z * transform_{i+1}
   4. add the transform of the EE due to rotation of the last joint

The code below compares CSDL FK to modern robotics FK. Good agreement (to within 0.001. Not sure where the discrepancy is coming from...) 

## TODOS:
- create a standalone test script to benchmark MR vs. CSDL kinematics
- I think twists are the correct way to represent rotations over rotation matrices 
- It seems like I am not performing my matrix logs correctly. The MR twist representing the final transform doesn't agree with the twist I calculate from CSDL, despite using the same input transformation matrix.
  - First, debug the rotation matrix log 
  - Then, debug the transformation matrix log
- If we are to use twists, the current FK function will need to be refactored. Should benchmark/validate the CSDL matrix exponentiation functions before I do this. 
- What happens when we hit edge cases? Like when R = I for the matrix log? 
- Why would a 4x4 transformation matrix give a 16x3 output? What is the interpretation of that? 

Test log:
- `trace`
- `vec_to_skew_symmetric`
- `skew_symmetric_to_vec`

In [24]:
import numpy as np
import csdl_alpha as csdl
from utils import *
import modern_robotics as mr

r = np.array((
    (0, -1, 0),
    (1, 0, 0),
    (0, 0, 1),
))
p = np.array((3,8,1))
t = np.identity(4)
t[:3, :3] = r
t[:3, -1] = p

mr_twist = mr.se3ToVec(mr.MatrixLog6(t))
mr_theta = np.linalg.norm(mr_twist[:3])
mr_screw = mr_twist / mr_theta

recorder.start()
t_csdl = csdl.Variable(value=t)
screw, theta = transform_log(t_csdl)
twist_csdl = screw * theta
jac = csdl.derivative(screw, theta)
recorder.stop()

print('csdl:\n', twist_csdl.value)
print('csdl jac:\n', np.round(jac.value, 3))
print('mr:\n', mr_twist)
print('mr jac:\n', mr.JacobianSpace((mr_screw, ), (mr_theta, )))
print(mr_screw)

csdl:
 [-0.          0.          1.57079633  8.6393798   3.92699082  1.        ]
csdl jac:
 [[ 0.   ]
 [ 0.   ]
 [-0.   ]
 [-1.5  ]
 [-4.   ]
 [-0.405]]
mr:
 [0.         0.         1.57079633 8.6393798  3.92699082 1.        ]
mr jac:
 [[0.         0.         1.         5.5        2.5        0.63661977]]
[0.         0.         1.         5.5        2.5        0.63661977]


In [25]:
import numpy as np
import csdl_alpha as csdl
from utils import *
import modern_robotics as mr


s1 = np.array([0, 0, 1, 0, 0, 0])
s2 = np.array([0, 0, 1, 0, -2, 0])
s3 = np.array([0, 0, 1, 0, -3, 0])

M = np.identity(4)
M[0, -1] = 3
s = np.array((s1, s2, s3)).T

angles = np.array((0, 0, 0))
# angles = np.random.random((3,)) * np.pi * 2 - np.pi


mr_fk = np.round(mr.FKinSpace(M, s, angles), 3)
print('MR:\n', np.round(mr_fk, 3))

recorder = csdl.Recorder(inline=True)

def calc_transform(euler_angles: csdl.Variable, translation: csdl.Variable):
    if euler_angles.size != 3:
        raise ValueError(f"Euler angle must be a 3 vector, but got {euler_angles.value}.")
    
    if translation.size != 3:
        raise ValueError(f"Translation must be a 3 vector, but got {translation.value}.")

    transform = csdl.Variable(value=np.identity(4))
    transform = transform.set(csdl.slice[0:3, -1], translation)

    rotation = euler_angle_to_rotation_matrix(euler_angles)
    transform = transform.set(csdl.slice[0:3, 0:3], rotation)
    return transform

## Following function uses transformation matrices. The FK itself works but:
## 1: the derivatives are not the right shape (16, 3)...
def forward_kinematics(angles, frames):
    # n = angles.size
    n = 3
    total_transform = csdl.Variable(value=np.identity(4), name='total_transform')
    ee_frame = csdl.Variable(value=np.identity(4))

    for i in csdl.frange(1, n):
        angle = angles[i-1]
        frame = frames.get(csdl.slice[:, :, i])
        total_transform = csdl.matmat(total_transform, csdl.matmat(rotation_matrix_to_transform(rotate_z(angle)), frame))
 
    total_transform = csdl.matmat(total_transform, csdl.matmat(rotation_matrix_to_transform(rotate_z(angles[-1])), ee_frame))
    return total_transform

def forward_kinematics_screw(screws, thetas, ee_frame):
    n = 3
    total_transform = csdl.Variable(value=np.identity(4), name='total_transform')

    # for i in csdl.frange(n):
    for i in range(n):
        theta = thetas[i]
        screw = screws.get(csdl.slice[:, i])
        total_transform = csdl.matmat(total_transform, transform_exp(screw, theta))

    total_transform = csdl.matmat(total_transform, ee_frame)
    return total_transform

def rotation_matrix_to_transform(rotation):
    trans = csdl.Variable(value=np.identity(4), name='temp_trans')
    trans = trans.set(csdl.slice[:3, :3], rotation)
    return trans

recorder.start()
# Frame 1
frame_1 = csdl.Variable(value=np.identity(4))

# Frame 2
frame_2 = csdl.Variable(value=np.identity(4))
frame_2 = frame_2.set(csdl.slice[:3, -1], np.array((2, 0, 0)))

# Frame 3 
frame_3 = csdl.Variable(value=np.identity(4))
frame_3 = frame_3.set(csdl.slice[:3, -1], np.array((1, 0, 0)))

frames = csdl.Variable(shape=(4, 4, 3), value=0)
frames = frames.set(csdl.slice[:, :, 0], frame_1)
frames = frames.set(csdl.slice[:, :, 1], frame_2)
frames = frames.set(csdl.slice[:, :, 2], frame_3)

theta = csdl.Variable(value=angles, name='theta')
csdl_fk = forward_kinematics(theta, frames)



p_jacobian = csdl.derivative(csdl_fk[:3, -1], theta)
out_angles = rotation_matrix_to_euler_angle(csdl_fk[:3, :3])
# w_jacobian = csdl.derivative(out_angles, theta)
# total_jacobian = csdl.derivative(csdl_fk, theta)


# s1, 
# screw, log_theta = transform_log(csdl_fk)
# twist = screw * log_theta
# twist_jaco = csdl.derivative(twist, theta)

# rot = euler_angle_to_rotation_matrix(angles)
# trans = csdl.Variable(value=np.identity(4))
# trans = trans.set(csdl.slice[:3, :3], rot)

# # Transform frame 2 based on rotation from frame 1
# mat3 = csdl.matmat(trans, mat2)

recorder.stop()
print('CSDL: \n', np.round(csdl_fk.value, 3))
print('Match: ', np.all(np.abs(csdl_fk.value - mr_fk) < 0.001))
print('out_angles', np.round(out_angles.value, 3))
print('p out', np.round(csdl_fk.value[:3, -1], 3))

print('------')
print('mr jaco: \n', np.round(mr.JacobianSpace(s, angles), 3))
print('p jaco\n', np.round(p_jacobian.value, 3))
# print('csdl twist', np.round(twist.value, 3))
print('mr twist', np.round(mr.se3ToVec(mr.MatrixLog6(mr_fk))))
# print('Diff:', csdl_fk.value - mr_fk)
# print(np.round(mat3.valu/e, 3))

MR:
 [[1. 0. 0. 3.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
CSDL: 
 [[1. 0. 0. 3.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
Match:  True
out_angles [ 0. -0.  0.]
p out [3. 0. 0.]
------
mr jaco: 
 [[ 0.  0.  0.]
 [ 0.  0.  0.]
 [ 1.  1.  1.]
 [ 0.  0.  0.]
 [ 0. -2. -3.]
 [ 0.  0.  0.]]
p jaco
 [[0. 0. 0.]
 [3. 1. 0.]
 [0. 0. 0.]]
mr twist [0. 0. 0. 3. 0. 0.]


Inverse Kinematics: 
1. FK on initial guess
2. e(theta) = x_D - FK(theta)
3. dFK(theta) * dt = e(theta)

Calculate e_pos = norm(FK(theta)[0:3, -1], - x_D.pos)
Calculate e_rot = desired rotation matrix. Norm of dot product between vectors 

Pseudo-inverse for tall mats:
(A^T A)^-1 A^T A 

A = J^T J
solve linear system Ax = b

In [26]:
r = np.array([
    [0, -1, 0],
    [1, 0, 0],
    [0, 0, 1j]
], dtype=np.complex64)

recorder.start()
rc = csdl.Variable(value=r)

logrc = csdl.log(rc)
recorder.stop()

logrc.value


c:\Users\zomog\miniconda3\envs\mae270\lib\site-packages\csdl_alpha\utils\inputs.py:16: ComplexWarning: Casting complex values to real discards the imaginary part
  value = value.astype(dtype)
c:\Users\zomog\miniconda3\envs\mae270\lib\site-packages\csdl_alpha\src\operations\log.py:72: RuntimeWarning: divide by zero encountered in log
  return np.log(x) / np.log(y)
c:\Users\zomog\miniconda3\envs\mae270\lib\site-packages\csdl_alpha\src\operations\log.py:72: RuntimeWarning: invalid value encountered in log
  return np.log(x) / np.log(y)


array([[-inf,  nan, -inf],
       [  0., -inf, -inf],
       [-inf, -inf, -inf]])

In [27]:
import numpy as np
import csdl_alpha as csdl
from utils import *
import modern_robotics as mr


def calc_ik_residual(theta, frames, goal_transform):
    transform = forward_kinematics(theta, frames)
    pos_err = csdl.norm(transform[:3, -1] - goal_transform[:3, -1])


    x_ang_err = 1 - csdl.vdot(transform[:3, 0], goal_transform[:3, 0])
    y_ang_err = 1 - csdl.vdot(transform[:3, 1], goal_transform[:3, 1])
    z_ang_err = 1 - csdl.vdot(transform[:3, 2], goal_transform[:3, 2])
    all_ang_err = csdl.Variable(shape=(3,), value=0)
    
    all_ang_err = all_ang_err.set(csdl.slice[0], x_ang_err)
    all_ang_err = all_ang_err.set(csdl.slice[1], y_ang_err)
    all_ang_err = all_ang_err.set(csdl.slice[2], z_ang_err)

    ang_err = csdl.norm(all_ang_err)
    return pos_err, ang_err

def calc_twist_err(transform, goal_transform):
    Tsb = transform
    Tsd = goal_transform

    Tbd = csdl.matmat(invert_transform(Tsb), Tsd)
    print('Tbd', np.round(Tbd.value, 3))
    screw_axis, theta = transform_log(Tbd)
    print('screw, theta', screw_axis.value, theta.value)
    twist = screw_axis * theta

    return csdl.matvec(adjoint(Tsb), twist)

def mr_twist_err(transform, goal):
    Vs = np.dot(mr.Adjoint(transform), \
                    mr.se3ToVec(mr.MatrixLog6(np.dot(mr.TransInv(transform), goal))))
    return Vs


def ik(theta, screws, ee_frame, goal_transform, pos_tol, ang_tol, max_iter=5):
    transform = forward_kinematics_screw(screws, theta, ee_frame)
    # pos_err, ang_err = calc_ik_residual(theta, frames, goal_transform)
    print('intial pos', np.round(transform.value, 3))
    print('goal', np.round(goal_transform.value, 3))
    err = calc_twist_err(transform, goal_transform)
    ang_err = csdl.norm(err[:3])
    pos_err = csdl.norm(err[3:])

    n_iter = 0

    while (pos_err.value > pos_tol or ang_err.value > ang_tol) and n_iter < max_iter:
        jac = calc_jacobian(screws, theta)

        jac_square = csdl.matmat(csdl.transpose(jac), jac)
        err_prime = csdl.matvec(csdl.transpose(jac), err)
        d_theta = csdl.solve_linear(jac_square, err_prime)
        print('d_theta', np.round(d_theta.value, 3))
        print(np.round(np.linalg.pinv(jac.value) @ err.value, 3))
        print('mathc?', np.all(np.round(d_theta.value, 3) - np.round(np.linalg.pinv(jac.value) @ err.value, 3) == 0))
        theta = theta + d_theta

        transform = forward_kinematics_screw(screws, theta, ee_frame)
        # pos_err, ang_err = calc_ik_residual(theta, frames, goal_transform)
        err = calc_twist_err(transform, goal_transform)

        print('csdl twist', np.round(err.value, 3))
        print('mr twist', np.round(mr_twist_err(transform.value, goal_transform.value), 3))
        ang_err = csdl.norm(err[:3])
        pos_err = csdl.norm(err[3:])


        
        n_iter += 1
    
    return theta

# try to get working with implicit vars. 
def ik2(theta, screws, ee_frame, goal_transform, pos_tol, ang_tol, max_iter=5):
    transform = forward_kinematics_screw(screws, theta, ee_frame)
    err = calc_twist_err(transform, goal_transform)
    
    jac = calc_jacobian(screws, theta)

    jac_square = csdl.matmat(csdl.transpose(jac), jac)
    err_prime = csdl.matvec(csdl.transpose(jac), err)

    solver = csdl.nonlinear_solvers.Newton()
    solver.add_state(theta, err_prime)

    # solver = csdl.nonlinear_solvers.Newton()
    # solver.add_state(theta, err)
    # e(theta) = xd - fk(theta)
    # e(theta + dt) = e(theta) - J(theta)*dt
    # J(theta)*dt = e(theta)
    # J^T(theta) J(theta)*dt = J^T(theta)e(theta)
    # dt = (J^T J)^-1 J^T e(theta)
    #   
    # d_theta = csdl.solve_linear(jac_square, err_prime)
    # theta = theta + d_theta

    solver.run()

    return theta
# err(theta) = desired - curr_pos(theta)
# err (6,); 
# theta is implicit
# err  is residual 
# 



angles = np.array((0.1, 0.1, 0.1))
M = np.identity(4)
M[0, -1] = 3
recorder = csdl.Recorder(inline=True)
recorder.start()
# Frame 1
frame_1 = csdl.Variable(value=np.identity(4))
cs1 = joint_transform_to_screw_axis(frame_1)

# Frame 2
frame_2 = csdl.Variable(value=np.identity(4))
frame_2 = frame_2.set(csdl.slice[:3, -1], np.array((2, 0, 0)))
cs2 = joint_transform_to_screw_axis(frame_2)

# Frame 3 
frame_3 = csdl.Variable(value=np.identity(4))
frame_3 = frame_3.set(csdl.slice[:3, -1], np.array((1, 0, 0)))


frames = csdl.Variable(shape=(4, 4, 3), value=0)
frames = frames.set(csdl.slice[:, :, 0], frame_1)
frames = frames.set(csdl.slice[:, :, 1], frame_2)
frames = frames.set(csdl.slice[:, :, 2], frame_3)

frame_3 = csdl.matmat(frame_2, frame_3)
cs3 = joint_transform_to_screw_axis(frame_3)

theta = csdl.ImplicitVariable(value=angles)

screws = csdl.Variable(shape=(6, 3), value=0)
screws = screws.set(csdl.slice[:, 0], cs1)
screws = screws.set(csdl.slice[:, 1], cs2)
screws = screws.set(csdl.slice[:, 2], cs3)
ee_frame = csdl.Variable(value=M)

fk_frame = forward_kinematics(theta, frames)
fk_screw = forward_kinematics_screw(screws, theta, ee_frame)
jac_analytical = csdl.derivative(fk_frame, theta)


# transform = forward_kinematics(theta, frames)


goal = np.identity(4)
goal[:3, -1] = np.array((2.9, 0, 0))
goal_transform = csdl.Variable(name='goal', value=goal)

# err = calc_twist_err(fk_screw, goal_transform)


# theta_new = ik(theta, screws, ee_frame, goal_transform, 0.001, 0.001)
theta_new = ik(theta, screws, ee_frame, goal_transform, 0.001, 0.001)
recorder.stop()
# print(np.round(fk_frame.value, 3))
# print(np.round(fk_screw.value, 3))
print('-----------')
print('Theta', np.round(theta.value, 5))
print('Theta_new', np.round(theta_new.value, 5))
print('mr theta:', mr.IKinSpace(s, M, goal, angles, 0.001, 0.001))
print('csdl jac\n', np.round(jac_analytical.value, 3))
# print('calc jac\n', np.round(jac_calc.value, 3))
print('mr jac\n', np.round(mr.JacobianSpace(s, angles), 3))
# print('state x1: ', theta.value)

# # The solved residuals
# print('residual x1: ', theta_new.value)
# # print('residual x2: ', ang_err.value)


IndexError: too many indices for array: array is 1-dimensional, but 3 were indexed

In [ ]:
from utils import *

def calc_jacobian_q(frames, theta, m):
    n = theta.size
    jac = csdl.Variable(value=np.zeros((7, n)))
    p_m = m[:3, -1]


    for i in csdl.frange(1, n):
        z = frames[:3, 2, i-1]
        p = frames[:3, 3, i-1]


        # Calculate the quaternions for the problem
        w = rotation_matrix_to_quat(frames[:, :, i])
        q = left_quat_multiply(w)
        # Pad rotation axis with additional dimension to multiply with q
        z_qvec = csdl.Variable(shape=(4,), value=0)
        z_qvec = z_qvec.set(csdl.slice[1:], z)

        # calculate orientation parts of the jacobian
        jac = jac.set(csdl.slice[:4, i], 0.5 * csdl.matmat(q, z_qvec))

        # calculate position parts of the jacobian
        jac = jac.set(csdl.slice[4:, i], csdl.cross(z, p_m - p))


def calc_7twist_err(transform, goal_transform):
    p_curr = transform[:3, -1]
    p_goal = transform[:3, -1]

    q_curr = rotation_matrix_to_quat(transform)
    q_goal = rotation_matrix_to_quat(goal_transform)

    p_err = p_goal - p_curr
    q_err = q_goal - q_curr

    err = csdl.Variable(shape=(7,), value=0)
    err = err.set(csdl.slice[:4], q_err)
    err = err.set(csdl.slice[4:p_err])
    return err

def ik3(theta, screws, ee_frame, goal_transform, pos_tol, ang_tol, max_iter=5):
    transform = forward_kinematics_screw(screws, theta, ee_frame)
    # pos_err, ang_err = calc_ik_residual(theta, frames, goal_transform)
    print('trans', np.round(transform.value, 3))
    print('trans', np.round(goal_transform.value, 3))
    err = calc_twist_err(transform, goal_transform)
    ang_err = csdl.norm(err[:3])
    pos_err = csdl.norm(err[3:])


    jac = calc_jacobian_q(frames, theta, ee_frame)
    n_iter = 0

    while (pos_err.value > pos_tol or ang_err.value > ang_tol) and n_iter < max_iter:
        

        jac_square = csdl.matmat(csdl.transpose(jac), jac)
        err_prime = csdl.matvec(csdl.transpose(jac), err)
        d_theta = csdl.solve_linear(jac_square, err_prime)
        print('d_theta', np.round(d_theta.value, 3))
        print(np.round(np.linalg.pinv(jac.value) @ err.value, 3))
        print('mathc?', np.all(np.round(d_theta.value, 3) - np.round(np.linalg.pinv(jac.value) @ err.value, 3) == 0))
        theta = theta + d_theta

        transform = forward_kinematics_screw(screws, theta, ee_frame)
        # pos_err, ang_err = calc_ik_residual(theta, frames, goal_transform)
        err = calc_twist_err(transform, goal_transform)

        print('csdl twist', np.round(err.value, 3))
        print('mr twist', np.round(mr_twist_err(transform.value, goal_transform.value), 3))
        ang_err = csdl.norm(err[:3])
        pos_err = csdl.norm(err[3:])


        
        n_iter += 1
    
    return theta

# err(theta) = desired - curr_pos(theta)
# err (6,); 
# theta is implicit
# err  is residual 



angles = np.array((0.1, 0.1, 0.1))
M = np.identity(4)
M[0, -1] = 3

recorder.start()
# Frame 1
frame_1 = csdl.Variable(value=np.identity(4))
cs1 = joint_transform_to_screw_axis(frame_1)

# Frame 2
frame_2 = csdl.Variable(value=np.identity(4))
frame_2 = frame_2.set(csdl.slice[:3, -1], np.array((2, 0, 0)))
cs2 = joint_transform_to_screw_axis(frame_2)

# Frame 3 
frame_3 = csdl.Variable(value=np.identity(4))
frame_3 = frame_3.set(csdl.slice[:3, -1], np.array((1, 0, 0)))


frames = csdl.Variable(shape=(4, 4, 3), value=0)
frames = frames.set(csdl.slice[:, :, 0], frame_1)
frames = frames.set(csdl.slice[:, :, 1], frame_2)
frames = frames.set(csdl.slice[:, :, 2], frame_3)

frame_3 = csdl.matmat(frame_2, frame_3)
cs3 = joint_transform_to_screw_axis(frame_3)

theta = csdl.Variable(value=angles)

screws = csdl.Variable(shape=(6, 3), value=0)
screws = screws.set(csdl.slice[:, 0], cs1)
screws = screws.set(csdl.slice[:, 1], cs2)
screws = screws.set(csdl.slice[:, 2], cs3)
ee_frame = csdl.Variable(value=M)

fk_frame = forward_kinematics(theta, frames)
fk_screw = forward_kinematics_screw(screws, theta, ee_frame)



# transform = forward_kinematics(theta, frames)


goal = np.identity(4)
goal[:3, -1] = np.array((2.9, 0, 0))
goal_transform = csdl.Variable(name='goal', value=goal)

theta_new = ik(theta, screws, ee_frame, goal_transform, 0.001, 0.001)

recorder.stop()
print(np.round(fk_frame.value, 3))
print(np.round(fk_screw.value, 3))
# print('state x1: ', theta.value)

# # The solved residuals
# print('residual x1: ', theta_new.value)
# # print('residual x2: ', ang_err.value)


TypeError: 'InverseKinematics' object is not callable

In [ ]:
import numpy as np

def lup_decomposition(A):
    n = A.shape[0]
    # Create an identity matrix to track permutations (P)
    P = np.eye(n)
    L = np.zeros((n, n))
    U = A.copy().astype(float)

    for i in range(n):
        # --- Partial Pivoting ---
        # Find the index of the largest element in the current column (below diagonal)
        pivot_row = np.argmax(abs(U[i:n, i])) + i
        
        if U[pivot_row, i] == 0:
            raise ValueError("Matrix is singular and cannot be decomposed.")

        # Swap rows in U and P
        U[[i, pivot_row]] = U[[pivot_row, i]]
        P[[i, pivot_row]] = P[[pivot_row, i]]
        # Swap rows in L (only the parts already computed)
        L[[i, pivot_row]] = L[[pivot_row, i]]

        # --- Standard LU logic ---
        L[i, i] = 1
        for j in range(i + 1, n):
            factor = U[j, i] / U[i, i]
            L[j, i] = factor
            U[j, i:] -= factor * U[i, i:]
            
    return P, L, U

# Example with a zero on the diagonal
A = np.array([
    [0, 1, 2],
    [1, 2, 1],
    [2, 7, 8]
])

P, L, U = lup_decomposition(A)
print("P @ L @ U equals A:\n", P.T @ L @ U) # Use P.T because P is the permutation

In [ ]:
import numpy as np
import csdl_alpha as csdl
from utils import *
import modern_robotics as mr


s1 = np.array([0, 0, 1, 0, 0, 0])
s2 = np.array([0, 0, 1, 0, -2, 0])
s3 = np.array([0, 0, 1, 0, -3, 0])

M = np.identity(4)
M[0, -1] = 3
s = np.array((s1, s2, s3)).T

angles = np.array((np.pi/3, 1*np.pi/3, 0*np.pi/1.3), dtype=float)
# angles = np.random.random((3,)) * np.pi * 2 - np.pi


mr_fk = np.round(mr.FKinSpace(M, s, angles), 3)
print('MR:\n', np.round(mr_fk, 3))

recorder = csdl.Recorder(inline=True)

def calc_jacobian(screws, theta):
    n = screws.shape[-1]
    jac = csdl.Variable(shape=screws.shape, value=0)
    jac = jac.set(csdl.slice[:, 0], screws[:, 0])

    t = csdl.Variable(value=np.identity(4))
    i = csdl.Variable(value=0, shape=(1,), name='i')
    for i in range(1, n):
        t = csdl.matmat(t, transform_exp(screws[:, i-1], theta[i-1]))
        jac = jac.set(csdl.slice[:, i], csdl.matvec(adjoint(t), screws[:, i]))
    
    return jac


recorder.start()
m = csdl.Variable(value=M)
# Frame 1
frame_1 = csdl.Variable(value=np.identity(4))
cs1 = joint_transform_to_screw_axis(frame_1)

# Frame 2
frame_2 = csdl.Variable(value=np.identity(4))
frame_2 = frame_2.set(csdl.slice[:3, -1], np.array((2, 0, 0)))
cs2 = joint_transform_to_screw_axis(frame_2)

# Frame 3 
frame_3 = csdl.Variable(value=np.identity(4))
frame_3 = frame_3.set(csdl.slice[:3, -1], np.array((1, 0, 0)))
frame_3 = csdl.matmat(frame_2, frame_3)
cs3 = joint_transform_to_screw_axis(frame_3)

theta = csdl.Variable(value=angles)

fk = csdl.matmat(transform_exp(cs1, theta[0]), csdl.matmat(transform_exp(cs2, theta[1]), transform_exp(cs3, theta[2])))
fk = csdl.matmat(fk, m)

screws = csdl.Variable(shape=(6, 3), value=0)
screws = screws.set(csdl.slice[:, 0], cs1)
screws = screws.set(csdl.slice[:, 1], cs2)
screws = screws.set(csdl.slice[:, 2], cs3)

jac = calc_jacobian(screws, theta)
recorder.stop()
# print('csdl fk\n', np.round(fk.value, 3))
print('csdl jac:\n', np.round(jac.value, 3))
print('mr jac\n', np.round(mr.JacobianSpace(s, angles), 3))
print('csdl screw\n', screws.value)
print('mr screw\n', s)

MR:
 [[-0.5   -0.866  0.     0.5  ]
 [ 0.866 -0.5    0.     2.598]
 [ 0.     0.     1.     0.   ]
 [ 0.     0.     0.     1.   ]]
csdl jac:
 [[ 0.     0.     0.   ]
 [ 0.     0.     0.   ]
 [ 1.     1.     1.   ]
 [ 0.     1.732  2.598]
 [ 0.    -1.    -0.5  ]
 [ 0.     0.     0.   ]]
mr jac
 [[ 0.     0.     0.   ]
 [ 0.     0.     0.   ]
 [ 1.     1.     1.   ]
 [ 0.     1.732  2.598]
 [ 0.    -1.    -0.5  ]
 [ 0.     0.     0.   ]]
csdl screw
 [[ 0.  0.  0.]
 [ 0.  0.  0.]
 [ 1.  1.  1.]
 [ 0.  0.  0.]
 [ 0. -2. -3.]
 [ 0.  0.  0.]]
mr screw
 [[ 0  0  0]
 [ 0  0  0]
 [ 1  1  1]
 [ 0  0  0]
 [ 0 -2 -3]
 [ 0  0  0]]


In [35]:
import numpy as np
import csdl_alpha as csdl
import modern_robotics as mr
from utils import *
import time


s1 = np.array([0, 0, 1, 0, 0, 0])
s2 = np.array([0, 0, 1, 0, -2, 0])
s3 = np.array([0, 0, 1, 0, -3, 0])

s = np.array((s1, s2, s3)).T

class InverseKinematics(csdl.CustomExplicitOperation):
    def __init__(self, joint_frames, ee_frame):
        super().__init__()
        self.joint_frames = joint_frames.value
        self.ee_frame = ee_frame.value

    def evaluate(self, inputs: csdl.VariableGroup):
        self.declare_input('theta', inputs.theta)
        self.declare_input('goal', inputs.goal)

        err = self.create_output('err', inputs.theta.shape)

        self.declare_derivative_parameters('err', 'goal', dependent=False)

        output = csdl.VariableGroup()
        output.err = err

        return output
    
    def compute(self, input_vals, output_vals):
        ## CSDL Based implementation (slow)
        # theta = csdl.Variable(value=input_vals['theta'])
        # goal = csdl.Variable(value=input_vals['goal'])

        # joint_frames = csdl.Variable(value=self.joint_frames)
        # ee_frame = csdl.Variable(value=self.ee_frame)

        # t = forward_kinematics_screw(joint_frames, theta, ee_frame)
        # err = calc_twist_err(t, goal)
        # jac = calc_jacobian(joint_frames, theta)
        # err = csdl.matmat(csdl.transpose(jac), err).value

        # output_vals['err'] = err

        ## Modern Robotics Implmentation (fast)
        theta = input_vals['theta']
        goal = input_vals['goal']

        t = mr.FKinSpace(self.ee_frame, self.joint_frames, theta)
        tbd = mr.TransInv(t) @ goal
        v_body = mr.se3ToVec(mr.MatrixLog6(tbd))
        v_space = mr.Adjoint(t) @ v_body
        jac = mr.JacobianSpace(self.joint_frames, theta)


        output_vals['err'] = jac.T @ v_space

    def compute_derivatives(self, inputs, outputs, derivatives):
        ## CSDL Based implementation (slow)
        # theta = csdl.Variable(value=inputs['theta'])

        # joint_frames = csdl.Variable(value=self.joint_frames)

        # jac = calc_jacobian(joint_frames, theta)
        # derivatives['err', 'theta'] = csdl.matmat(-csdl.transpose(jac), jac).value
        
        ## Modern Robotics Implementation (fast)
        theta = inputs['theta']
        jac = mr.JacobianSpace(self.joint_frames, theta)
        derivatives['err', 'theta'] = -jac.T @ jac



angles = np.array((0.1, 0.1, 0.1))
M = np.identity(4)
M[0, -1] = 3
recorder = csdl.Recorder(inline=True)
recorder.start()
# Frame 1
frame_1 = csdl.Variable(value=np.identity(4))
cs1 = joint_transform_to_screw_axis(frame_1)

# Frame 2
frame_2 = csdl.Variable(value=np.identity(4))
frame_2 = frame_2.set(csdl.slice[:3, -1], np.array((2, 0, 0)))
cs2 = joint_transform_to_screw_axis(frame_2)

# Frame 3 
frame_3 = csdl.Variable(value=np.identity(4))
frame_3 = frame_3.set(csdl.slice[:3, -1], np.array((1, 0, 0)))


frames = csdl.Variable(shape=(4, 4, 3), value=0)
frames = frames.set(csdl.slice[:, :, 0], frame_1)
frames = frames.set(csdl.slice[:, :, 1], frame_2)
frames = frames.set(csdl.slice[:, :, 2], frame_3)

frame_3 = csdl.matmat(frame_2, frame_3)
cs3 = joint_transform_to_screw_axis(frame_3)



screws = csdl.Variable(shape=(6, 3), value=0)
screws = screws.set(csdl.slice[:, 0], cs1)
screws = screws.set(csdl.slice[:, 1], cs2)
screws = screws.set(csdl.slice[:, 2], cs3)
ee_frame = csdl.Variable(value=M)

# transform = forward_kinematics(theta, frames)


goal = np.identity(4)
goal[:3, -1] = np.array((2.9, 0, 0))
goal_transform = csdl.Variable(name='goal', value=goal)



ik_input = csdl.VariableGroup()
theta = csdl.ImplicitVariable(value=angles)
ik_input.theta = theta
ik_input.goal = goal

ik = InverseKinematics(screws, ee_frame)

ik_err = ik.evaluate(ik_input)
solver = csdl.nonlinear_solvers.Newton()
start = time.time()
solver.add_state(theta, ik_err.err, tolerance=1e-3)
solver.run()

print('csdl ik', time.time() - start)

recorder.stop()

start = time.time()
for i in range(50):
    mr.IKinSpace(s, M, goal, angles, 0.001, 0.001)
print('mr ik', time.time() - start)



# Evaluate: Set up the model
#  
#
# Compute: 
# return the error
# 
# Compute Derivative 
# Return the derivative of the error twist (pseudo-inverse of Jacobian)? 
        

nonlinear solver: newton_nlsolver converged in 4 iterations.
csdl ik 0.005873918533325195
mr ik 0.10685586929321289


In [2]:
a = csdl.VariableGroup()

recorder.start()
test = csdl.Variable(value=32)
a.test = test

test = test + 42
recorder.stop()

print(a.test.value, test.value)


[32.] [74.]


In [ ]:
## UNIT TESTS

recorder.start()
v1 = csdl.Variable(value=np.array((4, 9, 1)))
r = csdl.Variable(value=np.arange(9).reshape((3,3)))

t = np.array((
    (0, -1, 0, 1),
    (1, 0, 0, 2),
    (0, 0, 1, 3),
    (0, 0, 0, 1)
))

t = csdl.Variable(value=t)



tr = trace(r)
ss = vec_to_skew_symmetric(v1)
v2 = skew_symmetric_to_vec(ss)
tinv = invert_transform(t)
print(csdl.arccos(csdl.Variable(value=1.01)).value)

recorder.stop()
print(v1.value)
print(r.value)
print(tr.value)
print(ss.value)
print(v2.value)
print(tinv.value)
print(mr.TransInv(t.value))

[nan]
[4. 9. 1.]
[[0. 1. 2.]
 [3. 4. 5.]
 [6. 7. 8.]]
[12.]
[[ 0. -1.  9.]
 [ 1.  0. -4.]
 [-9.  4.  0.]]
[4. 9. 1.]
[[ 0.  1.  0. -2.]
 [-1.  0.  0.  1.]
 [ 0.  0.  1. -3.]
 [ 0.  0.  0.  1.]]
[[ 0.  1.  0. -2.]
 [-1.  0.  0.  1.]
 [ 0.  0.  1. -3.]
 [ 0.  0.  0.  1.]]


c:\Users\zomog\miniconda3\envs\mae270\lib\site-packages\csdl_alpha\src\operations\trig.py:67: RuntimeWarning: invalid value encountered in arccos
  return np.arccos(x)
